## Before you start:
**Tools → Settings → Editor → completions / suggestions / linting → disable**

In [2]:
import sys
import gc
import weakref

**Task 1:** Study how basic reference counting works.
Create a string and investigate its reference count at different stages. Return a tuple: (count_after_creation, count_after_ref, count_after_del)

In [68]:
def task1():
    text = "Hello, world!"

    count_after_creation = sys.getrefcount(text)

    # Reference to the same object
    ref_text = text

    count_after_ref = sys.getrefcount(text)

    del ref_text

    count_after_del = sys.getrefcount(text)

    return (count_after_creation, count_after_ref, count_after_del)

# Check
print("Task 1:", task1())

Task 1: (4, 5, 4)


**Task 2:** Function impact on Reference Counting
Create a list and pass it to a function. Compare reference count before and after function call. Return a tuple: (count_before_call, count_during_call, count_after_call)

In [52]:
def task2():
      def process_list(lst):
          return sys.getrefcount(lst)

      lst = [0, 1, 2, 3, 4, 5]

      count_before_call = sys.getrefcount(lst)
      count_during_call = process_list(lst)
      count_after_call = sys.getrefcount(lst)

      return (count_before_call, count_during_call, count_after_call)

# Check
print("Task 2:", task2())

Task 2: (2, 3, 2)


**Task 3:** Cyclic references and Memory Leaks
Create two objects with a cyclic reference, then break it. Use gc to check the number of collected objects. Return the number of objects collected by gc after breaking the reference.

In [69]:
def task3():
    class Node:
        def __init__(self, name):
            self.name = name
            self.ref = None

    gc.collect()

    # Create cycle
    a = Node('a')
    b = Node('b')
    a.ref = b
    b.ref = a

    # Delete the reference
    del a
    del b

    collect = gc.collect()
    
    return collect

print("Task 3:", task3())

Task 3: 2


**Task 4:** Comparing Reference Count for different data types
Compare reference count for numbers, strings, lists and dictionaries.Return a dictionary with reference count for each type after creation.

In [70]:
def task4():
    # Hint: you can check int, str, list and others
    data = {
        'int': 123456789,
        'float': 3.14159,
        'list': [0, 1, 2, 3, 4, 5],
        'str': "Hello, world! " * 2,
        'dict': {'a': 1, 'b': 2, 'c': 3},
        'tuple': (0, 1, 2)
    }
    result = dict()

    for t, value in data.items():
        result[t] = sys.getrefcount(value)

    return result

print("Task 4:", task4())

Task 4: {'int': 6, 'float': 6, 'list': 4, 'str': 5, 'dict': 4, 'tuple': 5}


**Task 5:** Weak References
Create two objects with a weak reference between them. Ensure the objects can be deleted by the garbage collector. Return True if the weakref does not increase reference count.

In [71]:
def task5():
    class Data:
        def __init__(self, value):
            self.value = value

    data1 = Data(1)
    data2 = Data(2)

    count_before = sys.getrefcount(data1)

    weak = weakref.ref(data1)
    data2.ref = weak

    count_after = sys.getrefcount(data1)

    del data1
    del data2

    gc.collect()

    return (weak() is None) and (count_before == count_after)

print("Task 5:", task5())

Task 5: True


**Task 6:** Monitoring the Garbage Collector.
Register a callback function to track GC activity. Return a list of events tracked by the callback.

In [72]:
def task6():
    def callback(phase, info):
        events.append((phase, dict(info)))

    events = []
    
    # Register callback
    gc.callbacks.append(callback)

    gc.collect()
    gc.collect(1)
    gc.collect(2)

    # Unregister callback
    gc.callbacks.remove(callback)

    return events

print("Task 6:", '\n'.join(map(str, task6())), sep='\n')

Task 6:
('start', {'generation': 2, 'collected': 0, 'uncollectable': 0})
('stop', {'generation': 2, 'collected': 6, 'uncollectable': 0})
('start', {'generation': 1, 'collected': 0, 'uncollectable': 0})
('stop', {'generation': 1, 'collected': 0, 'uncollectable': 0})
('start', {'generation': 2, 'collected': 0, 'uncollectable': 0})
('stop', {'generation': 2, 'collected': 0, 'uncollectable': 0})


**Task 7:** GC Generation Analysis
Create objects and trace their movement between GC generations.Return a tuple with the number of objects in each generation before and after object creation.

In [232]:
def task7():
    gc.collect()

    # (gen 0, gen 1, gen 2)
    before_objs = gc.get_count()

    # Create many objects to populate gen 0
    objs = []
    for _ in range(500000):
        objs.append(object())

    gc.collect()

    # (gen 0, gen 1, gen 2)
    after_objs = gc.get_count()

    return before_objs, after_objs


print("Task 7:", task7())

Task 7: ((3, 0, 0), (7, 0, 0))


**Task 8:** Monitoring Garbage Collection Thresholds
Study how GC generation counters change when creating objects.Return a dictionary with the state of the counters before and after object creation.

In [14]:
def task8():
    initial_threshold = gc.get_threshold()

    gc.collect()

    initial_count = gc.get_count()

    # Create many objects to bump gen 0 counters
    objs = []
    for _ in range(500000):
        objs.append(object())

    after_creation_count = gc.get_count()

    collected = gc.collect(0)

    after_collect_count = gc.get_count()

    return {
        'thresholds': initial_threshold,
        'initial_count': initial_count,
        'after_creation': after_creation_count,
        'after_collect_gen0': after_collect_count,
        'collected_objects': collected
    }
    

print("Task 8:", '\n'.join(map(str, task8().items())), sep='\n')

Task 8:
('thresholds', (700, 10, 10))
('initial_count', (15, 0, 0))
('after_creation', (12, 0, 0))
('after_collect_gen0', (0, 1, 0))
('collected_objects', 0)


**Task 9:** Quick GC check.
Check if the GC collects cyclic references. Return the difference in the number of objects before and after collection.

In [243]:
def task9():
    a, b, c = [], [], []

    # Cycle: a -> b -> c -> a
    a.append(b)
    b.append(c)
    c.append(a)  

    count_before = len(gc.get_objects())
    collected = gc.collect()
    count_after = len(gc.get_objects())

    # (difference, collected objects)
    return (count_before - count_after, collected)

print("Task 9:", task9())

Task 9: (12, 3)


**Task 10:** Detector for "undying" objects
Find objects that survive forced garbage collection.Output the number of such objects.

In [21]:
def task10():
    gc.garbage.clear()
    gc.collect()

    gc_debug = gc.get_debug()
    gc.set_debug(gc_debug | gc.DEBUG_SAVEALL)

    class Uncollectable:
        def __init__(self, name):
            self.name = name
            self.ref = None
        
        def __del__(self):
            pass

    # Create cycle:
    a = Uncollectable('a')
    b = Uncollectable('b')
    a.ref = b
    b.ref = a

    del a
    del b

    gc.collect()

    result = len(gc.garbage)

    gc.garbage.clear()
    gc.set_debug(gc_debug)

    return result

print("Task 10:", task10())

Task 10: 2
